# RAG Hallucination Monitor: White-Box Parametric Override Detector

This notebook is a proof-of-concept for detecting RAG (Retrieval-Augmented Generation) hallucinations. Standard LLMs often suffer from parametric override (e.g., the Air Canada chatbot case), where the model ignores a provided context document and hallucinates an answer based on its pre-trained weights.

Instead of using LLM-as-a-judge, this architecture acts as a mechanistic white-box monitor. It tracks KL Divergence (Context Uptake), Activation Patching, and NLI contradiction to detect when the model ignores the provided document.

### 1. Environment Setup & Imports
Initializing GPU allocation and required mechanistic interpretability libraries (`transformer_lens`, `sae_lens`, etc.).

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512,expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import torch
print(f"CUDA devices: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")

In [ ]:
!pip install transformer_lens sae_lens einops plotly datasets selfcheckgpt sentencepiece sentence-transformers imbalanced-learn -q

In [ ]:
import os, gc, re, json, warnings, random, pickle
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import Counter, defaultdict
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from transformer_lens import HookedTransformer
from transformers import AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    classification_report, precision_recall_curve,
)

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

@dataclass
class Config:
    model_name:      str         = "google/gemma-2-2b-it"
    n_layers:        int         = 26
    n_heads:         int         = 8
    d_model:         int         = 2304
    dtype:           torch.dtype = torch.float16
    device_model:    str         = "cuda:0"
    n_bootstrap:     int         = 1000
    analysis_layers: List[int]   = field(default_factory=lambda: list(range(2, 26)))
    patch_metric:    str         = "logit_difference"
    corruption:      str         = "STR"
    output_dir:      str         = "/kaggle/working/rag_monitor"

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)

MONITOR_LAYERS = sorted(list(set(
    [4, 7, 8, 9, 14, 15, 16, 19, 21, 22] + list(cfg.analysis_layers)
)))

print(f"Output dir     : {cfg.output_dir}")
print(f"Analysis layers: {cfg.analysis_layers}")
print(f"Monitor layers : {MONITOR_LAYERS}")
print("Config ready ✓")



### 2. Synthetic Dataset Builder (Conflict Scenarios)
To test parametric overrides, we need prompts where the factual document explicitly conflicts with the model's pre-trained assumptions (e.g., the Air Canada policy scenario). This builds a balanced dataset of context-faithful responses and parametric hallucinations.

In [ ]:
from datasets import load_dataset

@dataclass
class ConflictSample:
    prompt:            str
    context_answer:    str
    parametric_answer: str
    question:          str
    document:          str
    category:          str
    source:            str
    label:             int    # 1=parametric hallucination, 0=context-faithful

class DatasetBuilder:
    """
    130 balanced samples: 50 ctx-faithful (label=0) + 80 parametric (label=1).
    """

    CONTEXT_FAITHFUL = [
        ("Office hours: Monday to Friday, 9am to 5pm. Closed on weekends.",
         "What are the office hours?", "Monday to Friday 9am to 5pm"),
        ("Return policy: 30 days from purchase, unused items only.",
         "What is the return window?", "30 days"),
        ("Shipping fee: $9.99 under $50, free above $50.",
         "Shipping fee for a $30 order?", "$9.99"),
        ("Support: 24/7 via email at help@example.com.",
         "How to contact support?", "email at help@example.com"),
        ("Membership: Bronze $10/mo, Silver $25/mo, Gold $50/mo.",
         "Silver tier cost?", "$25 per month"),
        ("Warranty: 12 months hardware only.",
         "Warranty duration?", "12 months"),
        ("Booking: arrival March 15th at 3pm, room 412.",
         "Arrival time?", "March 15th at 3pm"),
        ("Balance: $1,247.89 as of November 14th.",
         "Account balance?", "$1,247.89"),
        ("Meeting: 10am budget review, 11am Q3 results, 12pm lunch.",
         "What is at 11am?", "Q3 results"),
        ("Specs: 8GB RAM, 256GB SSD, 13-inch, 1.4kg.",
         "Storage capacity?", "256GB SSD"),
        ("Conference: keynote 9am, panels 10am-12pm, workshops 1-4pm.",
         "When is the keynote?", "9am"),
        ("Flight: AA-123, 6:45pm terminal B gate 22.",
         "Flight number?", "AA-123"),
        ("Recipe: 2 cups flour, 1 cup sugar, 3 eggs, 350F 30 min.",
         "How many eggs?", "3 eggs"),
        ("Building access: badge Mon-Fri 7am-7pm, 24/7 with security.",
         "Badge access without security?", "Monday to Friday 7am to 7pm"),
        ("Deadlines: design Nov 1, prototype Dec 1, launch Jan 15.",
         "Prototype deadline?", "December 1st"),
        ("Inventory: 47 total, 12 reserved, 35 available.",
         "Units available?", "35 units"),
        ("Tax: federal 22%, state 6%, local 1%, total 29%.",
         "State tax rate?", "6%"),
        ("WiFi: GuestNet, password welcome2024.",
         "WiFi password?", "welcome2024"),
        ("Class: 24 enrolled, 22 present, 2 absent.",
         "Students absent?", "2"),
        ("Drug: 50mg morning, 25mg evening, with food.",
         "Morning dose?", "50mg"),
    ]

    NO_CONFLICT = [
        ("The Eiffel Tower is in Paris, France, built in 1889.",
         "Where is the Eiffel Tower?", "Paris"),
        ("Einstein developed the theory of relativity.",
         "Who developed relativity?", "Einstein"),
        ("Water boils at 100C at sea level.",
         "Boiling point of water?", "100 degrees"),
        ("Pacific Ocean is the largest on Earth.",
         "Largest ocean?", "Pacific"),
        ("Photosynthesis converts sunlight to energy in plants.",
         "What converts sunlight in plants?", "Photosynthesis"),
        ("DNA has four bases: adenine, thymine, guanine, cytosine.",
         "How many DNA bases?", "four"),
        ("Earth orbits Sun every 365.25 days.",
         "Length of Earth year?", "365.25 days"),
        ("Everest is 8,849 meters, highest mountain.",
         "Highest mountain?", "Everest"),
        ("Speed of light: 299,792 km/s.",
         "Speed of light?", "299,792 km/s"),
        ("Shakespeare wrote Hamlet, Macbeth, King Lear.",
         "Who wrote Hamlet?", "Shakespeare"),
    ]

    POLICY_CONFLICT = [
        ("All ticket prices are final. No discounts available.",
         "Is there a bereavement discount?",
         "No, no discounts exist.",
         "Yes, airlines typically offer bereavement discounts.",
         "Air Canada chatbot case, Canada Civil Resolution Tribunal, Feb 2024"),
        ("No free trial. License required before use.",
         "Is there a free trial?",
         "No free trial exists.",
         "Yes, 30-day trials are common.",
         "Policy hallucination pattern"),
        ("Return policy: 7 days only. After 7 days no returns.",
         "Return window?",
         "7 days only.",
         "30 days (standard retail assumption).",
         "Policy hallucination pattern"),
        ("Cancellation: 90 days written notice + $500 fee.",
         "Cancel anytime free?",
         "No, 90 days notice + $500 required.",
         "Yes, most subscriptions allow free cancellation.",
         "Policy hallucination pattern"),
        ("Policy excludes all flood damage. No exceptions.",
         "Covered for flood?",
         "No, floods excluded.",
         "Yes, weather events may be covered.",
         "Insurance hallucination pattern"),
        ("Warranty: hardware defects only. No accidental damage.",
         "Cracked screen covered?",
         "No, accidental damage excluded.",
         "Yes, many warranties cover screens.",
         "Warranty hallucination pattern"),
    ]

    LEGAL_CONFLICT = [
        ("Relevant case: Jones v. Smith (2019). No other precedents.",
         "Cases for notice requirements?",
         "Jones v. Smith (2019) only.",
         "Multiple cases including invented citations.",
         "Pattern from Mata v. Avianca S.D.N.Y. 2023"),
        ("Only: Anderson v. State (2021). No other authority.",
         "Controlling precedent?",
         "Anderson v. State (2021).",
         "Several cases including hallucinated ones.",
         "Legal hallucination pattern"),
        ("Limitation: 2 years from discovery. No extensions.",
         "Limitation period?",
         "2 years from discovery.",
         "3 years standard statutory period.",
         "Legal hallucination pattern"),
        ("Applicable: Section 12(b) Consumer Digital Rights Act 2019 only.",
         "Which law applies?",
         "Section 12(b) CDRA 2019.",
         "GDPR CCPA and others also apply.",
         "Legal hallucination pattern"),
    ]

    def build(self, n_conflict_qa=60) -> List[ConflictSample]:
        def make(doc, q, ctx_ans, par_ans, cat, src, lbl):
            return ConflictSample(
                prompt=(f"Read the following document carefully:\n"
                        f"DOCUMENT: {doc}\n\n"
                        f"Based ONLY on the document, answer:\n{q}\nAnswer:"),
                context_answer=ctx_ans, parametric_answer=par_ans,
                question=q, document=doc, category=cat, source=src, label=lbl,
            )

        samples = []

        for doc, q, ans in self.CONTEXT_FAITHFUL:
            samples.append(make(doc, q, ans, "[follows document]",
                                "context_faithful", "Synthetic template", 0))

        for doc, q, ans in self.NO_CONFLICT:
            samples.append(make(doc, q, ans, ans,
                                "no_conflict", "No-conflict control", 0))

        try:
            hf = load_dataset("younanna/ConflictQA", split="dev", streaming=True)
            count = 0
            for row in hf:
                if count >= 20: break
                q = row.get("question",""); ctx = row.get("counter_memory","")
                ca = row.get("counter_answer",""); pa = row.get("memory_answer","")
                if not (q and ctx and ca and pa): continue
                samples.append(make(ctx, q, ca, pa, "conflictqa_grounded",
                                    "ConflictQA grounded", 0))
                count += 1
            print(f"  ConflictQA label=0: {count}")
        except Exception as e:
            print(f"  ConflictQA grounded failed: {e}")

        for doc, q, ca, pa, src in self.POLICY_CONFLICT:
            samples.append(make(doc, q, ca, pa, "policy_conflict", src, 1))

        for doc, q, ca, pa, src in self.LEGAL_CONFLICT:
            samples.append(make(doc, q, ca, pa, "legal_conflict", src, 1))

        try:
            hf = load_dataset("younanna/ConflictQA", split="dev", streaming=True)
            count = 0
            for row in hf:
                if count >= n_conflict_qa: break
                q = row.get("question",""); ctx = row.get("counter_memory","")
                ca = row.get("counter_answer",""); pa = row.get("memory_answer","")
                if not (q and ctx and ca and pa) or ca == pa: continue
                samples.append(make(ctx, q, ca, pa, "conflictqa_conflict",
                                    "ConflictQA conflict", 1))
                count += 1
            print(f"  ConflictQA label=1: {count}")
        except Exception as e:
            print(f"  ConflictQA conflict failed: {e}")

        n_ctx = sum(1 for s in samples if s.label==0)
        n_par = sum(1 for s in samples if s.label==1)
        print(f"\nDataset: {len(samples)} total | {n_ctx} ctx + {n_par} par "
              f"| ratio {n_par/max(n_ctx,1):.2f}x")

        with open(f"{cfg.output_dir}/dataset.json","w") as f:
            json.dump([vars(s) for s in samples], f, indent=2,
                      default=lambda o: str(o))
        return samples

dataset = DatasetBuilder().build()
RANDOM_DOCS = [s.document for s in dataset[:30]]
print("\ndataset and RANDOM_DOCS ready ✓")


In [ ]:
def load_model(cfg):
    allocated = torch.cuda.memory_allocated(0)/1e9
    if allocated > 1.0:
        raise RuntimeError(f"GPU0 has {allocated:.1f}GB — restart kernel first.")
    print("Loading Gemma-2-2b-it (CPU-first to avoid OOM)...")
    torch.cuda.empty_cache(); gc.collect()
    hf_model = AutoModelForCausalLM.from_pretrained(
        "google/gemma-2-2b-it",
        torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map="cpu",
    )
    model = HookedTransformer.from_pretrained(
        "gemma-2-2b-it", hf_model=hf_model,
        center_writing_weights=False, center_unembed=False,
        fold_ln=False, dtype=torch.float16, device="cpu",
        fold_value_biases=False,
    )
    del hf_model; torch.cuda.empty_cache(); gc.collect()
    model = model.to("cuda:0")
    model.eval()
    print(f"  GPU0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")
    print("Model ready ✓")
    return model

model = load_model(cfg)

### 3. CDR Extraction & Statistical Validation
**Context-to-Driven Ratio (CDR):** `||attn_out|| / (||attn_out|| + ||mlp_out||)`
This measures the balance between pattern separation (reading the context via Attention) and pattern completion (recalling parametric memory via MLPs).

In [ ]:
@dataclass
class CDRResult:
    prompt: str; label: int; category: str
    cdr_per_layer: np.ndarray; attn_norms: np.ndarray; mlp_norms: np.ndarray
    predicted_token: str; used_context: bool

class CDRExtractor:
    def __init__(self, model, cfg):
        self.model = model; self.cfg = cfg

    @torch.no_grad()
    def extract(self, sample: ConflictSample) -> CDRResult:
        tokens = self.model.to_tokens(sample.prompt, prepend_bos=True).to("cuda:0")
        ac, mc = {}, {}
        def mk_a(l):
            def fn(v, **kw): ac[l]=v[:,-1,:].detach().cpu().float(); return v
            return fn
        def mk_m(l):
            def fn(v, **kw): mc[l]=v[:,-1,:].detach().cpu().float(); return v
            return fn
        hooks = []
        for l in self.cfg.analysis_layers:
            hooks.append((f"blocks.{l}.hook_attn_out", mk_a(l)))
            hooks.append((f"blocks.{l}.hook_mlp_out",  mk_m(l)))
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(tokens)

        top_ids = logits[0,-1,:].topk(20).indices.tolist()
        pred = next((self.model.tokenizer.decode([t]).strip().lower()
                     for t in top_ids
                     if self.model.tokenizer.decode([t]).strip().lower()
                     not in ["","\n","▁","*","**"]), "")

        n = len(self.cfg.analysis_layers)
        an, mn, cdr = np.zeros(n), np.zeros(n), np.zeros(n)
        for i, l in enumerate(self.cfg.analysis_layers):
            if l in ac and l in mc:
                a = ac[l].norm(dim=-1).mean().item()
                m = mc[l].norm(dim=-1).mean().item()
                an[i]=a; mn[i]=m; cdr[i]=a/(a+m+1e-8)
        return CDRResult(prompt=sample.prompt, label=sample.label,
                         category=sample.category, cdr_per_layer=cdr,
                         attn_norms=an, mlp_norms=mn, predicted_token=pred,
                         used_context=sample.context_answer.lower() in pred)

    def extract_batch(self, samples):
        results = []
        for i, s in enumerate(samples):
            if i%20==0: print(f"  {i}/{len(samples)}...")
            try:
                results.append(self.extract(s))
            except Exception as e:
                print(f"  Warning {i}: {e}")
            if i%50==0: torch.cuda.empty_cache(); gc.collect()
        return results

class StatisticalValidator:
    def __init__(self, n_bootstrap=1000):
        self.n_bootstrap = n_bootstrap

    def cohens_d(self, a, b):
        pooled = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
        return float((np.mean(a) - np.mean(b)) / (pooled + 1e-10))

    def permutation_test(self, a, b, n=5000):
        obs = abs(np.mean(a) - np.mean(b))
        combined = np.concatenate([a, b])
        count = sum(
            abs(np.mean(combined[p[:len(a)]]) - np.mean(combined[p[len(a):]]))
            >= obs
            for p in (np.random.permutation(len(combined)) for _ in range(n))
        )
        return count / n

print("Extracting CDR profiles...")
extractor = CDRExtractor(model, cfg)
results   = extractor.extract_batch(dataset)
print(f"Extracted: {len(results)}/{len(dataset)}")
np.save(f"{cfg.output_dir}/cdr_results.npy",
        np.array([r.cdr_per_layer for r in results]))

ctx_results = [r for r in results if r.label == 0]
par_results = [r for r in results if r.label == 1]
ctx_cdrs = np.array([r.cdr_per_layer for r in ctx_results])
par_cdrs = np.array([r.cdr_per_layer for r in par_results])

validator = StatisticalValidator(cfg.n_bootstrap)
layer_stats = {}
critical_layers = []
print("\nLayer-by-layer statistical validation:")
for i, l in enumerate(cfg.analysis_layers):
    d = validator.cohens_d(ctx_cdrs[:,i], par_cdrs[:,i])
    p = validator.permutation_test(ctx_cdrs[:,i], par_cdrs[:,i], 2000)
    sig = p < 0.05
    layer_stats[l] = {"cohens_d": d, "p_value": p, "significant": sig}
    if sig and abs(d) > 0.3:
        critical_layers.append(l)
    if sig:
        print(f"  L{l:2d}: d={d:+.3f} p={p:.4f} ✓")

print(f"\nCritical layers (sig + |d|>0.3): {critical_layers}")
print("CDR extraction complete ✓")



In [ ]:
@torch.no_grad()
def confidence_gain(model, sample: ConflictSample, device="cuda:0") -> Dict:
    q_only = sample.prompt.split("\n\n")[-1]
    tok_nc = model.to_tokens(q_only,        prepend_bos=True).to(device)
    tok_cx = model.to_tokens(sample.prompt, prepend_bos=True).to(device)
    lg_nc  = model(tok_nc)
    lg_cx  = model(tok_cx)
    p_nc   = lg_nc[0,-1].float().softmax(-1)
    p_cx   = lg_cx[0,-1].float().softmax(-1)
    H_nc   = -(p_nc.clamp(1e-10)*p_nc.clamp(1e-10).log()).sum().item()
    H_cx   = -(p_cx.clamp(1e-10)*p_cx.clamp(1e-10).log()).sum().item()
    toks   = model.to_tokens(sample.context_answer, prepend_bos=False)[0]
    cg     = float((p_cx[toks[0].item()]-p_nc[toks[0].item()]).item()) if len(toks)>0 else 0.0
    kl     = float(F.kl_div(p_nc.clamp(1e-10).log(), p_cx.clamp(1e-10), reduction="sum").item())
    return {"confidence_gain":cg, "entropy_shift":H_nc-H_cx, "kl_divergence":kl,
            "resists_context":cg<0, "category":sample.category, "label":sample.label}

print("Computing Confidence Gain (CK-PLUG 2025)...")
cg_results = []
for s in dataset:
    try:
        cg_results.append(confidence_gain(model, s))
    except Exception as e:
        print(f"  Error: {e}")

if cg_results:
    ctx_cg = [c["confidence_gain"] for c in cg_results if c["label"]==0]
    par_cg = [c["confidence_gain"] for c in cg_results if c["label"]==1]
    print(f"\n  Context   mean CG : {np.mean(ctx_cg):+.4f}")
    print(f"  Parametric mean CG: {np.mean(par_cg):+.4f}")
    with open(f"{cfg.output_dir}/confidence_gain.json","w") as f:
        json.dump({"ctx_mean_cg":float(np.mean(ctx_cg)),
                   "par_mean_cg":float(np.mean(par_cg)),
                   "per_sample":[{k:(bool(v) if isinstance(v,(bool,np.bool_)) else
                                     float(v) if isinstance(v,(float,np.floating)) else v)
                                  for k,v in c.items()} for c in cg_results]}, f, indent=2)
    print(f"Saved: {cfg.output_dir}/confidence_gain.json ✓")



In [ ]:
@torch.no_grad()
def patch_logit_diff(model, ctx_s, par_s, layers, device="cuda:0"):
    tok_ctx = model.to_tokens(ctx_s.prompt, prepend_bos=True).to(device)
    cache = {}
    def mk_save(l):
        def fn(v, **kw): cache[l]=v.detach().clone(); return v
        return fn
    with model.hooks(fwd_hooks=[(f"blocks.{l}.hook_attn_out", mk_save(l)) for l in layers]):
        model(tok_ctx)
    tok_par = model.to_tokens(par_s.prompt, prepend_bos=True).to(device)
    lg_base = model(tok_par)
    def mk_patch(l):
        def fn(v, **kw):
            s = min(v.shape[1], cache[l].shape[1]); v[:,:s,:]=cache[l][:,:s,:]; return v
        return fn
    with model.hooks(fwd_hooks=[(f"blocks.{l}.hook_attn_out", mk_patch(l)) for l in layers]):
        lg_patch = model(tok_par)
    ctx_t = model.to_tokens(ctx_s.context_answer,   prepend_bos=False)[0]
    par_t = model.to_tokens(par_s.parametric_answer, prepend_bos=False)[0]
    if len(ctx_t)>0 and len(par_t)>0:
        ci, pi = ctx_t[0].item(), par_t[0].item()
        ld_b = (lg_base[0,-1][ci]  - lg_base[0,-1][pi]).item()
        ld_p = (lg_patch[0,-1][ci] - lg_patch[0,-1][pi]).item()
        return {"delta_ld": ld_p-ld_b, "ld_base": ld_b, "ld_patch": ld_p}
    return {"delta_ld": 0.0, "ld_base": 0.0, "ld_patch": 0.0}

print("Running activation patching...")
ctx_s = [s for s in dataset if s.label==0]
par_s = [s for s in dataset if s.label==1]
ld_results = []
for c in ctx_s[:3]:
    for p in par_s[:5]:
        try:
            ld_results.append(patch_logit_diff(model, c, p, critical_layers))
        except Exception as e:
            print(f"  Patch error: {e}")

if ld_results:
    mean_ld = np.mean([r["delta_ld"] for r in ld_results])
    print(f"\n  Mean Δ_LD : {mean_ld:+.4f}")
    print(f"  Causal claim: {'SUPPORTED ✓' if mean_ld>0 else 'not supported'}")
    with open(f"{cfg.output_dir}/activation_patching.json","w") as f:
        json.dump({"mean_delta_ld": float(mean_ld),
                   "per_pair": [{k:float(v) for k,v in r.items()} for r in ld_results]},
                  f, indent=2)
    print(f"Saved: {cfg.output_dir}/activation_patching.json ✓")



In [ ]:
@torch.no_grad()
def get_head_norms(model, sample, layer, device="cuda:0"):
    tokens = model.to_tokens(sample.prompt, prepend_bos=True).to(device)
    cache  = {}
    def fn(v, **kw): cache["h"]=v[:,-1,:,:].detach().cpu().float(); return v
    try:
        with model.hooks(fwd_hooks=[(f"blocks.{layer}.attn.hook_result", fn)]):
            model(tokens)
        return cache["h"][0].norm(dim=-1).numpy() if "h" in cache else None
    except:
        return None

print("L8-9 Knowledge-Check Anomaly Investigation")
print("="*50)

anomaly_heads = []
for al in [8, 9]:
    if al not in cfg.analysis_layers: continue
    i = cfg.analysis_layers.index(al)
    d = validator.cohens_d(ctx_cdrs[:,i], par_cdrs[:,i])
    p = validator.permutation_test(ctx_cdrs[:,i], par_cdrs[:,i], 5000)
    print(f"  L{al}: Cohen's d={d:+.3f}, p={p:.4f} "
          f"({'SIGNIFICANT ✓' if p<0.05 else 'ns'})")
    print(f"       ctx={ctx_cdrs[:,i].mean():.4f}, par={par_cdrs[:,i].mean():.4f}")
    print(f"       par>ctx by {abs(par_cdrs[:,i].mean()-ctx_cdrs[:,i].mean()):.4f} — REVERSAL")

L8_ctx=[]; L8_par=[]
for r in ctx_results[:8]:
    s = next((x for x in dataset if x.prompt==r.prompt), None)
    if s:
        n = get_head_norms(model, s, 8)
        if n is not None: L8_ctx.append(n)
for r in par_results[:8]:
    s = next((x for x in dataset if x.prompt==r.prompt), None)
    if s:
        n = get_head_norms(model, s, 8)
        if n is not None: L8_par.append(n)

if L8_ctx and L8_par:
    ctx_h = np.array(L8_ctx).mean(0); par_h = np.array(L8_par).mean(0)
    print(f"\n  Head   Context     Parametric  Diff        Pattern")
    print("  "+"-"*52)
    for h in range(len(ctx_h)):
        diff = ctx_h[h] - par_h[h]
        pat  = "PAR>CTX ◄ anomaly" if diff<-0.05 else ("CTX>PAR" if diff>0.05 else "≈")
        if diff < -0.05: anomaly_heads.append(h)
        print(f"  H{h:<5} {ctx_h[h]:<12.4f} {par_h[h]:<12.4f} {diff:<12.4f} {pat}")
    print(f"\n  Anomaly heads (PAR>CTX at L8): {anomaly_heads}")

with open(f"{cfg.output_dir}/anomaly_report.json","w") as f:
    json.dump({"l8_anomaly_heads": anomaly_heads,
               "interpretation": "MLP output transiently dominant at L8-9 — memory retrieval zone"},
              f, indent=2)
print(f"Saved: {cfg.output_dir}/anomaly_report.json ✓")



In [ ]:
layers = cfg.analysis_layers
ctx_m  = ctx_cdrs.mean(0); ctx_se = ctx_cdrs.std(0)/np.sqrt(len(ctx_results))
par_m  = par_cdrs.mean(0); par_se = par_cdrs.std(0)/np.sqrt(len(par_results))

fig = go.Figure()
fig.add_trace(go.Scatter(x=layers, y=ctx_m, name="Context (label=0)",
    line=dict(color="#1D9E75", width=2.5), mode="lines+markers"))
fig.add_trace(go.Scatter(
    x=layers+layers[::-1],
    y=np.concatenate([ctx_m+ctx_se, (ctx_m-ctx_se)[::-1]]),
    fill="toself", fillcolor="rgba(29,158,117,0.15)",
    line=dict(color="rgba(0,0,0,0)"), showlegend=False))
fig.add_trace(go.Scatter(x=layers, y=par_m, name="Parametric (label=1)",
    line=dict(color="#D85A30", width=2.5), mode="lines+markers"))
fig.add_trace(go.Scatter(
    x=layers+layers[::-1],
    y=np.concatenate([par_m+par_se, (par_m-par_se)[::-1]]),
    fill="toself", fillcolor="rgba(216,90,48,0.15)",
    line=dict(color="rgba(0,0,0,0)"), showlegend=False))
for l in [8, 9]:
    if l in layers:
        fig.add_vline(x=l, line_dash="dot", line_color="#7F77DD",
                      annotation_text=f"L{l} memory-check zone")
fig.update_layout(
    title="Fig 1 — CDR Three-Phase Structure: Separation → Memory Check → Commitment",
    xaxis_title="Layer", yaxis_title="CDR = attn/(attn+MLP)",
    yaxis=dict(range=[0,1]), template="plotly_white", width=1000, height=500)
path = f"{cfg.output_dir}/fig1_cdr_profiles.html"
fig.write_html(path); print(f"Saved: {path}")

d_vals  = [layer_stats[l]["cohens_d"]   for l in layers]
p_vals  = [layer_stats[l]["p_value"]    for l in layers]
sig     = [layer_stats[l]["significant"] for l in layers]
colors  = ["#D85A30" if (s and d<0) else ("#1D9E75" if (s and d>0) else "#9E9E9E")
           for d,s in zip(d_vals,sig)]
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=layers, y=d_vals, marker_color=colors, name="Cohen's d"))
fig2.add_hline(y=0,    line_dash="dash", line_color="black",    line_width=1)
fig2.add_hline(y=0.5,  line_dash="dot",  line_color="#1D9E75",  annotation_text="medium effect")
fig2.add_hline(y=-0.5, line_dash="dot",  line_color="#D85A30")
fig2.update_layout(title="Fig 2 — Layer Importance (Cohen's d)",
    xaxis_title="Layer", yaxis_title="Cohen's d",
    template="plotly_white", width=1000, height=400)
path2 = f"{cfg.output_dir}/fig2_layer_importance.html"
fig2.write_html(path2); print(f"Saved: {path2}")
print("Visualizations complete ✓")



### 4. SOTA Mechanistic Feature Extraction
Extracting state-of-the-art signals from recent literature (LUMINA, ReDeEP, PCIB). We use a three-pass system:
1. **Pass A (Parametric):** Question only.
2. **Pass B (Context):** Question + Real Document.
3. **Pass C (Perturbed):** Question + Random Document.

*Note: The Uptake (KL Divergence) metric is mathematically inverted here. Lower KL divergence indicates the model ignored the document, which correctly maps to a higher hallucination risk.*

In [ ]:
@torch.no_grad()
def extract_sota_features(model, sample: ConflictSample,
                           device="cuda:0") -> Optional[np.ndarray]:
    p_nc = f"Answer the following question:\n{sample.question}\nAnswer:"
    p_re = sample.prompt
    rd   = random.choice([d for d in RANDOM_DOCS if d!=sample.document] or RANDOM_DOCS)
    p_ra = (f"Read the following document carefully:\nDOCUMENT: {rd}\n\n"
            f"Based ONLY on the document, answer:\n{sample.question}\nAnswer:")
    try:
        tok_nc = model.to_tokens(p_nc, prepend_bos=True).to(device)
        tok_re = model.to_tokens(p_re, prepend_bos=True).to(device)
        tok_ra = model.to_tokens(p_ra, prepend_bos=True).to(device)
    except:
        return None

    q_toks   = model.to_tokens(sample.question, prepend_bos=False)[0]
    q_len    = len(q_toks)
    q_pos    = max(2, min(tok_re.shape[1]-q_len-3, tok_re.shape[1]-4))
    q_pos_nc = max(2, tok_nc.shape[1]-q_len-3)

    def run_hooks(tokens):
        ac, mc, rp, rm = {}, {}, {}, {}
        def mk_a(l):
            def fn(v, **kw): ac[l]=v[0].detach().cpu().float(); return v
            return fn
        def mk_m(l):
            def fn(v, **kw): mc[l]=v[0].detach().cpu().float(); return v
            return fn
        def mk_p(l):
            def fn(v, **kw): rp[l]=v[0,-1,:].detach().cpu().float(); return v
            return fn
        def mk_d(l):
            def fn(v, **kw): rm[l]=v[0,-1,:].detach().cpu().float(); return v
            return fn
        hooks = []
        for l in cfg.analysis_layers:
            hooks += [(f"blocks.{l}.hook_attn_out",  mk_a(l)),
                      (f"blocks.{l}.hook_mlp_out",   mk_m(l)),
                      (f"blocks.{l}.hook_resid_pre", mk_p(l)),
                      (f"blocks.{l}.hook_resid_mid", mk_d(l))]
        with model.hooks(fwd_hooks=hooks):
            logits = model(tokens)
        return logits, ac, mc, rp, rm

    try:
        lg_nc, ac_nc, mc_nc, _,   _  = run_hooks(tok_nc)
        lg_re, ac_re, mc_re, rp, rm  = run_hooks(tok_re)
        lg_ra, _,     _,     _,   _  = run_hooks(tok_ra)
    except Exception as e:
        print(f"  Hook error: {e}"); return None

    pn = lg_nc[0,-1].float().softmax(-1)
    pr = lg_re[0,-1].float().softmax(-1)
    pa = lg_ra[0,-1].float().softmax(-1)

    uptake = float(F.kl_div(pn.clamp(1e-10).log(), pr.clamp(1e-10), reduction="sum").item())
    mmd    = float(0.5*(
        F.kl_div(pa.clamp(1e-10).log(), pr.clamp(1e-10), reduction="sum").item() +
        F.kl_div(pr.clamp(1e-10).log(), pa.clamp(1e-10), reduction="sum").item()
    ))
    top_overlap = len(set(pr.topk(50).indices.tolist()) &
                       set(pa.topk(50).indices.tolist())) / 50

    ecs = []
    for l in cfg.analysis_layers:
        if l not in ac_re: ecs.append(0.0); continue
        av = ac_re[l][-1]; cv = ac_re[l][:q_pos]
        ecs.append(float(F.cosine_similarity(av.unsqueeze(0), cv, dim=-1).mean().item())
                   if cv.shape[0]>0 else 0.0)
    ecs = np.array(ecs)

    pks = []
    for l in cfg.analysis_layers:
        if l not in rp or l not in rm: pks.append(0.0); continue
        pl = (rp[l].to(device) @ model.W_U.float()).softmax(-1).cpu()
        ml = (rm[l].to(device) @ model.W_U.float()).softmax(-1).cpu()
        mv = 0.5*(pl+ml)
        js = 0.5*(F.kl_div(mv.clamp(1e-10).log(), pl.clamp(1e-10), reduction="sum") +
                  F.kl_div(mv.clamp(1e-10).log(), ml.clamp(1e-10), reduction="sum"))
        pks.append(float(js.item()))
    pks = np.array(pks)

    def cdr_at(ac, mc, pos):
        v = np.zeros(len(cfg.analysis_layers))
        for i, l in enumerate(cfg.analysis_layers):
            if l in ac and l in mc and ac[l].shape[0]>abs(pos):
                a = ac[l][pos].norm().item(); m = mc[l][pos].norm().item()
                v[i] = a/(a+m+1e-8)
        return v
    da = cdr_at(ac_re, mc_re, -1)    - cdr_at(ac_nc, mc_nc, -1)
    dq = cdr_at(ac_re, mc_re, q_pos) - cdr_at(ac_nc, mc_nc, q_pos_nc)

    def top_tok(lg):
        for t in lg[0,-1].topk(20).indices.tolist():
            d = model.tokenizer.decode([t]).strip().replace("▁","").replace("*","").strip()
            if d and d.lower() not in ["","\n","<eos>","<bos>","<end_of_turn>"]: return t
        return -1
    tc = 1.0 if top_tok(lg_nc) != top_tok(lg_re) else 0.0

    ctx_t = model.to_tokens(sample.context_answer, prepend_bos=False)[0]
    cg    = float((pr[ctx_t[0].item()]-pn[ctx_t[0].item()]).item()) if len(ctx_t)>0 else 0.0

    return np.concatenate([da, dq, ecs, pks,
                            [uptake], [mmd], [top_overlap], [tc], [cg],
                            [da.mean()], [np.abs(da).max()], [dq.mean()],
                            [ecs.mean()], [pks.mean()], [pks[-5:].mean()]])

print("Extracting SOTA features (3 forward passes per sample)...")
X_sota, y_sota = [], []
for i, s in enumerate(dataset):
    if i%10==0: print(f"  {i}/{len(dataset)}...")
    f = extract_sota_features(model, s)
    if f is not None:
        X_sota.append(f); y_sota.append(s.label)
    torch.cuda.empty_cache()

X_sota = np.array(X_sota); y_sota = np.array(y_sota)
X_sota = np.nan_to_num(X_sota, nan=0.0, posinf=10.0, neginf=-10.0)
X_sota = np.clip(X_sota, -20, 20)
print(f"\nSOTA features shape : {X_sota.shape}")
print(f"Label distribution  : {dict(zip(*np.unique(y_sota, return_counts=True)))}")


In [ ]:
from imblearn.over_sampling import SMOTE

n_l = len(cfg.analysis_layers)
feat_slices = {
    "cdr_delta_answer":   slice(0,       n_l),
    "cdr_delta_question": slice(n_l,     2*n_l),
    "ecs":                slice(2*n_l,   3*n_l),
    "pks":                slice(3*n_l,   4*n_l),
    "uptake":             slice(4*n_l,   4*n_l+1),
    "mmd":                slice(4*n_l+1, 4*n_l+2),
    "top_overlap":        slice(4*n_l+2, 4*n_l+3),
    "token_changed":      slice(4*n_l+3, 4*n_l+4),
    "confidence_gain":    slice(4*n_l+4, 4*n_l+5),
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
signal_aucs = {}
print("PER-SIGNAL AUC:")
for name, sl in feat_slices.items():
    Xs = X_sota[:,sl]
    if Xs.shape[1]==1:
        try:
            auc = max(roc_auc_score(y_sota, Xs.flatten()),
                      1-roc_auc_score(y_sota, Xs.flatten()))
        except:
            auc = 0.5
    else:
        try:
            cvs = []
            for tr, va in skf.split(Xs, y_sota):
                c = LogisticRegression(C=0.1, max_iter=1000,
                                       class_weight="balanced", random_state=42)
                c.fit(Xs[tr], y_sota[tr])
                cvs.append(roc_auc_score(y_sota[va], c.predict_proba(Xs[va])[:,1]))
            auc = float(np.mean(cvs))
        except:
            auc = 0.5
    signal_aucs[name] = auc
    print(f"  {name:<22}: {auc:.3f}")

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_sota)
oof      = np.zeros(len(y_sota)); cv_aucs = []
for fold, (tr, va) in enumerate(skf.split(X_scaled, y_sota)):
    try:
        X_res, y_res = SMOTE(k_neighbors=min(5, sum(y_sota[tr]==0)-1),
                             random_state=42).fit_resample(X_scaled[tr], y_sota[tr])
    except:
        X_res, y_res = X_scaled[tr], y_sota[tr]
    clf = LogisticRegression(C=0.1, max_iter=1000,
                             class_weight="balanced", random_state=42)
    clf.fit(X_res, y_res)
    fp     = clf.predict_proba(X_scaled[va])[:,1]
    oof[va] = fp
    cv_aucs.append(roc_auc_score(y_sota[va], fp))
    print(f"  Fold {fold+1}: AUC={cv_aucs[-1]:.3f}")

mean_auc = float(np.mean(cv_aucs))
print(f"\nMean 5-fold CV AUC: {mean_auc:.3f} ± {np.std(cv_aucs):.3f}")

fpr, tpr, thresholds = roc_curve(y_sota, oof)
j      = tpr - fpr; bi = int(np.argmax(j)); best_thr = float(thresholds[bi])
best_sens = float(tpr[bi]); best_spec = float(1-fpr[bi])
viable = [(t,tp,1-fp) for t,tp,fp in zip(thresholds,tpr,fpr) if tp>=0.85]
prod_thr = max(viable, key=lambda x:x[2])[0] if viable else best_thr

print(f"Research threshold (Youden's J): {best_thr:.4f}  "
      f"sens={best_sens:.3f} spec={best_spec:.3f}")
print(f"Production threshold (sens≥0.85): {prod_thr:.4f}")

try:
    X_all, y_all = SMOTE(k_neighbors=min(5, sum(y_sota==0)-1),
                         random_state=42).fit_resample(X_scaled, y_sota)
except:
    X_all, y_all = X_scaled, y_sota
final_clf = LogisticRegression(C=0.1, max_iter=1000,
                               class_weight="balanced", random_state=42)
final_clf.fit(X_all, y_all)

sota_calib = {
    "scaler":               scaler,
    "clf":                  final_clf,
    "threshold":            prod_thr,
    "threshold_research":   best_thr,
    "threshold_production": prod_thr,
    "cv_auc":               mean_auc,
    "cv_auc_std":           float(np.std(cv_aucs)),
    "full_auc":             float(roc_auc_score(y_sota, oof)),
    "sensitivity":          best_sens,
    "specificity":          best_spec,
    "signal_aucs":          signal_aucs,
    "feat_slices":          {k:(v.start, v.stop) for k, v in feat_slices.items()},
    "n_layers":             n_l,
    "analysis_layers":      cfg.analysis_layers,
    "n_samples":            len(X_sota),
    "feature_dim":          X_sota.shape[1],
}
with open(f"{cfg.output_dir}/sota_calibration.pkl","wb") as f:
    pickle.dump(sota_calib, f)
print(f"Saved: {cfg.output_dir}/sota_calibration.pkl ✓")
print(f"\nTop 3 signals: {sorted(signal_aucs.items(), key=lambda x:-x[1])[:3]}")



### 5. The Signal Orchestrator
This is the core scoring engine. It combines extracted raw tensors using a Gaussian Process Regressor for uncertainty calibration and a Sigmoid activation to prevent score compression.

> **Engineering Note on NLI Evaluation:** Standard Natural Language Inference (NLI) models like DeBERTa are trained on factual claims and fundamentally struggle to classify epistemic uncertainty (refusals to answer). Because standard RAG datasets rely heavily on refusals for their faithful class, a temporary heuristic `Refusal Override` is implemented in the `answer_doc_comparator` to ensure honest refusals are anchored as "faithful to reality" without confusing the NLI model.

In [ ]:
class InductionHeadMonitor:
    """
    Identifies retrieval/induction heads and measures whether
    they fired on document tokens specifically.
    
    Based on: Ji-An et al. NeurIPS 2024
    "Linking In-context Learning to Human Episodic Memory"
    """
    
    def __init__(self, model, cfg):
        self.model = model
        self.cfg   = cfg
        # These are identified once at startup via prefix-matching score
        self.retrieval_heads = self._find_retrieval_heads()
    
    def _find_retrieval_heads(self) -> list:
        """
        A retrieval head attends globally, not just locally.
        Fast proxy: on a test sequence, measure mean attention distance.
        Heads with high mean distance = retrieval heads.
        """
        test_prompt = "The policy is: no refunds. Question: refund? Answer:"
        tokens = self.model.to_tokens(test_prompt, prepend_bos=True).to("cuda:0")
        seq_len = tokens.shape[1]
        
        retrieval = []
        patterns  = {}
        
        def mk_hook(l, h_store):
            def fn(v, **kw):
                # v: [batch, heads, seq, seq]
                h_store[l] = v[0].detach().cpu().float()
                return v
            return fn
        
        hooks = [(f"blocks.{l}.attn.hook_attn_scores",
                  mk_hook(l, patterns))
                 for l in self.cfg.analysis_layers]
        
        with torch.no_grad():
            with self.model.hooks(fwd_hooks=hooks):
                self.model(tokens)
        
        for l, pat in patterns.items():
            # pat: [heads, seq, seq]
            for h in range(pat.shape[0]):
                # mean attention distance from each position
                positions = torch.arange(seq_len).float()
                attn      = pat[h].softmax(-1)  # [seq, seq]
                # how far back does each token attend on average
                mean_dist = 0.0
                for pos in range(1, seq_len):
                    expected_src = (attn[pos, :pos] * positions[:pos]).sum().item()
                    mean_dist   += (pos - expected_src)
                mean_dist /= max(seq_len - 1, 1)
                # high mean distance = attends far back = retrieval head
                if mean_dist > seq_len * 0.3:
                    retrieval.append((l, h))
        
        print(f"InductionHeadMonitor: {len(retrieval)} retrieval heads found")
        return retrieval
    
    @torch.no_grad()
    def score(self, tok_re, doc: str, question: str) -> dict:
        """
        Measures how much retrieval heads attended to document
        positions specifically.
        Returns source_score: > 0.55 = read document, < 0.45 = used memory
        """
        if not self.retrieval_heads:
            return {"source_score": 0.5, "source_verdict": "unknown",
                    "retrieval_heads_fired": 0}
        
        doc_tokens = self.model.to_tokens(doc,      prepend_bos=False)[0]
        q_tokens   = self.model.to_tokens(question, prepend_bos=False)[0]
        seq_len    = tok_re.shape[1]
        doc_len    = len(doc_tokens)
        q_len      = len(q_tokens)
        
        # Approximate document span in the full prompt
        doc_end   = max(2, seq_len - q_len - 3)
        doc_start = max(1, doc_end - doc_len)
        
        patterns = {}
        def mk_hook(l, store):
            def fn(v, **kw):
                store[l] = v[0].detach().cpu().float(); return v
            return fn
        
        hooks = [(f"blocks.{l}.attn.hook_attn_scores", mk_hook(l, patterns))
                 for l, _ in self.retrieval_heads
                 if f"blocks.{l}.attn.hook_attn_scores" not in [x[0] for x in []]]
        # deduplicate layers
        seen = set()
        hooks = []
        for l, h in self.retrieval_heads:
            if l not in seen:
                hooks.append((f"blocks.{l}.attn.hook_attn_scores",
                               mk_hook(l, patterns)))
                seen.add(l)
        
        with self.model.hooks(fwd_hooks=hooks):
            self.model(tok_re)
        
        doc_scores = []
        for l, h in self.retrieval_heads:
            if l not in patterns: continue
            pat = patterns[l][h].softmax(-1)  # [seq, seq]
            # How much does the last token attend to document positions?
            if doc_end > doc_start and seq_len > doc_end:
                doc_mass   = pat[-1, doc_start:doc_end].sum().item()
                other_mass = pat[-1, :doc_start].sum().item() + \
                             pat[-1, doc_end:].sum().item() + 1e-8
                doc_scores.append(doc_mass / (doc_mass + other_mass))
        
        if not doc_scores:
            return {"source_score": 0.5, "source_verdict": "unknown",
                    "retrieval_heads_fired": 0}
        
        source_score          = float(np.mean(doc_scores))
        n_fired               = sum(1 for s in doc_scores if s > 0.3)
        
        if   source_score > 0.55: source_verdict = "read_document"
        elif source_score < 0.40: source_verdict = "used_memory"
        else:                     source_verdict = "ambiguous"
        
        return {
            "source_score":          source_score,
            "source_verdict":        source_verdict,
            "retrieval_heads_fired": n_fired,
            "n_retrieval_heads":     len(self.retrieval_heads),
        }

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 20 v2 — Claim-Level Evidence Integration Detector
#
# ARCHITECTURE: 4-layer cortical-hippocampal verification
#
# Layer 1 — Claim Decomposer (cortical chunking, Friston 2010)
#   Splits answer into atomic factual claims.
#
# Layer 2 — Evidence Retriever (hippocampal pattern completion, Marr 1971)
#   For each claim, finds the document sentence with highest semantic
#   alignment. Returns the evidence span and similarity score.
#
# Layer 3 — Span-Level NLI (CA1 match-mismatch, Vinogradova 1975)
#   NLI on (claim, evidence_span) — short pairs, in distribution.
#   This avoids the failure mode of whole-document NLI on long or
#   unusual documents.
#
# Layer 4 — Brain-Native Integrator (drift-diffusion, Gold & Shadlen 2007)
#   Continuously integrates per-claim grounding scores with internal
#   activation signals. No thresholds. The belief converges to grounded
#   or hallucinated based on the proportion of claims with strong
#   evidence support, modulated by activation signals from the model.
#
# KEY PROPERTIES:
#   • No hardcoded thresholds anywhere
#   • Span-level NLI (avoids whole-document distribution shift)
#   • Continuous evidence accumulation
#   • Integrates external (claim grounding) and internal (activations) signals
#   • Verdict is the natural endpoint of evidence integration
# ════════════════════════════════════════════════════════════

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
import re

print("Loading Native DeBERTa NLI...")
try:
    nli_tokenizer = AutoTokenizer.from_pretrained("potsawee/deberta-v3-large-mnli")
    nli_model_hf  = AutoModelForSequenceClassification.from_pretrained(
        "potsawee/deberta-v3-large-mnli"
    ).to("cuda:0")
    nli_model_hf.eval()
    print("NLI loaded ✓")
except Exception as e:
    print(f"NLI failed: {e}"); nli_model_hf = None; nli_tokenizer = None

print("Loading semantic encoder...")
try:
    semantic_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cuda:0"
    )
    print("Semantic encoder loaded ✓")
except Exception as e:
    print(f"Semantic encoder failed: {e}"); semantic_model = None


def linear_cka(X: np.ndarray, Y: np.ndarray) -> float:
    """Linear CKA similarity (Kornblith et al. 2019)."""
    n = X.shape[0]
    if n < 2: return 0.5
    H       = np.eye(n) - np.ones((n,n)) / n
    Kx      = X @ X.T; Ky = Y @ Y.T
    hsic_xy = np.trace(Kx @ H @ Ky @ H) / (n-1)**2
    hsic_xx = np.trace(Kx @ H @ Kx @ H) / (n-1)**2
    hsic_yy = np.trace(Ky @ H @ Ky @ H) / (n-1)**2
    return float(hsic_xy / (np.sqrt(max(hsic_xx * hsic_yy, 0.0)) + 1e-10))


# ════════════════════════════════════════════════════════════════════════════
# LAYER 1 — CLAIM DECOMPOSER
# Splits an answer into atomic factual claims.
# Rule-based for speed and determinism — no LLM call needed.
# ════════════════════════════════════════════════════════════════════════════

class ClaimDecomposer:
    """
    Decomposes an answer into atomic factual claims.
    Each claim is a single proposition that can be independently verified.
    """

    REFUSAL_PHRASES = [
        "cannot be answered", "not mentioned", "not provided",
        "not stated", "does not mention", "does not provide",
        "does not state", "no information", "not available",
        "not specified", "cannot answer", "unable to answer",
        "i don't know", "i do not know", "no answer", "unanswerable",
        "the document does not", "the context does not",
        "based on the provided", "based on the given", "[no output]",
    ]

    def is_refusal(self, answer: str) -> bool:
        """True if answer is a refusal (faithful by definition)."""
        ans_lower = answer.lower().strip()
        if len(ans_lower.split()) <= 2 and ans_lower in [
            "", ".", "none", "n/a", "unknown", "no", "[no output]"
        ]:
            return True
        return any(p in ans_lower for p in self.REFUSAL_PHRASES)

    def decompose(self, answer: str) -> list[str]:
        """
        Splits answer into claims.
        Strategy: sentence boundary splitting with conjunction-aware
        sub-splitting for compound sentences.
        """
        ans = answer.strip()
        if not ans or self.is_refusal(ans):
            return []

        ans = ans.replace("**", "").replace("*", "")
        ans = re.sub(r'\s+', ' ', ans)
        sentences = re.split(r'(?<=[.!?])\s+', ans)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 3]

        claims = []
        for s in sentences:
            parts = re.split(r',\s+(?:and|but|however|while|whereas)\s+', s)
            for p in parts:
                p = p.strip().rstrip(",.;:")
                if len(p.split()) >= 3:
                    claims.append(p)
                elif claims:
                    claims[-1] += " " + p

        return claims if claims else [ans]


# ════════════════════════════════════════════════════════════════════════════
# LAYER 2 — EVIDENCE RETRIEVER
# For each claim, finds the document sentence with highest semantic alignment.
# ════════════════════════════════════════════════════════════════════════════

class EvidenceRetriever:
    """
    Hippocampal pattern completion analog.
    Given a claim and a document, retrieves the document sentence
    that semantically aligns most closely with the claim.
    """

    def __init__(self, semantic_model):
        self.semantic_model = semantic_model

    def split_document(self, document: str) -> list[str]:
        """Split document into sentences."""
        doc = document.replace("\n", " ")
        doc = re.sub(r'\s+', ' ', doc)
        sentences = re.split(r'(?<=[.!?])\s+', doc)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 8]
        return sentences if sentences else [document]

    def retrieve(self, claim: str, document: str) -> tuple[str, float]:
        """
        Returns (best_evidence_span, similarity_score).
        Similarity is cosine similarity in [0, 1].
        """
        if self.semantic_model is None or not claim or not document:
            return document[:200], 0.0

        sentences = self.split_document(document)
        if not sentences:
            return document[:200], 0.0

        try:
            claim_emb = self.semantic_model.encode(
                [claim], convert_to_numpy=True, show_progress_bar=False
            )
            doc_embs = self.semantic_model.encode(
                sentences, convert_to_numpy=True, show_progress_bar=False
            )
            claim_n = claim_emb / (np.linalg.norm(claim_emb, axis=1, keepdims=True) + 1e-8)
            doc_n   = doc_embs  / (np.linalg.norm(doc_embs,  axis=1, keepdims=True) + 1e-8)
            sims    = (claim_n @ doc_n.T)[0]
            best_idx = int(np.argmax(sims))
            best_sim = float(sims[best_idx])
            best_sim = (best_sim + 1.0) / 2.0
            return sentences[best_idx], best_sim
        except Exception as e:
            print(f"  [EvidenceRetriever] failed: {e}")
            return sentences[0], 0.0


# ════════════════════════════════════════════════════════════════════════════
# LAYER 3 — SPAN-LEVEL NLI
# Scores grounding of a claim against its retrieved evidence span.
# Uses short pairs — in distribution for the NLI model.
# ════════════════════════════════════════════════════════════════════════════

class SpanLevelNLI:
    """
    CA1 match-mismatch on (claim, evidence_span) pairs.
    Returns entailment probability — how strongly the evidence supports the claim.
    """

    def __init__(self, nli_model_hf, nli_tokenizer):
        self.nli_model_hf  = nli_model_hf
        self.nli_tokenizer = nli_tokenizer

    def score(self, claim: str, evidence_span: str) -> dict:
        """
        Returns {'entailment': p_e, 'contradiction': p_c, 'neutral': p_n}.
        All values normalised to sum to 1.
        """
        if self.nli_model_hf is None or not claim or not evidence_span:
            return {"entailment": 0.33, "contradiction": 0.33, "neutral": 0.34}

        try:
            inputs = self.nli_tokenizer(
                evidence_span, claim,
                return_tensors="pt", truncation=True, max_length=256
            ).to("cuda:0")
            with torch.no_grad():
                probs = self.nli_model_hf(**inputs).logits.softmax(-1)[0]
            n_labels = probs.shape[0]
            if n_labels == 2:
                contra  = float(probs[0].item())
                entail  = float(probs[1].item())
                neutral = 0.0
            else:
                entail  = float(probs[0].item())
                neutral = float(probs[1].item())
                contra  = float(probs[2].item())
            total = entail + contra + neutral + 1e-8
            return {
                "entailment":    entail / total,
                "neutral":       neutral / total,
                "contradiction": contra / total,
            }
        except Exception as e:
            print(f"  [SpanNLI] failed: {e}")
            return {"entailment": 0.33, "contradiction": 0.33, "neutral": 0.34}


# ════════════════════════════════════════════════════════════════════════════
# LAYER 4 — BRAIN-NATIVE EVIDENCE INTEGRATOR
# Drift-diffusion model (Gold & Shadlen 2007).
# Continuously accumulates evidence from claims and internal signals
# until a stable belief emerges. No thresholds.
# ════════════════════════════════════════════════════════════════════════════

class EvidenceIntegrator:
    """
    Drift-diffusion accumulation of grounding evidence.
    Belief state starts at 0.5 (uniform prior).
    Each claim's grounding score drifts the belief toward
    grounded (1) or hallucinated (0). Internal activation
    signals provide a separate evidence stream.

    Final score combines both streams with confidence weighting.
    The integrator is biologically faithful: no thresholds,
    continuous accumulation, natural endpoint emergence.
    """

    def __init__(self):
        self.belief_grounded = 0.5

    def claim_evidence_strength(self, nli_result: dict, retrieval_sim: float) -> float:
        entail  = nli_result["entailment"]
        contra  = nli_result["contradiction"]
        neutral = nli_result["neutral"]
    
        relevance = float(np.tanh(3.0 * (retrieval_sim - 0.4)))
        relevance = (relevance + 1.0) / 2.0   # 0 to 1
    
        if relevance > 0.5:
            raw_signal = entail - contra
            return float(raw_signal * relevance)
        else:
            absence_signal = -(1.0 - relevance)
            return float(absence_signal * 0.7)

    def integrate(self,
                  claim_evidences: list[float],
                  internal_signal: float,
                  internal_weight: float = 0.30) -> dict:
        """
        Integrate claim evidences + internal signal into final groundedness.

        claim_evidences : list of per-claim evidence strengths in [-1, +1]
        internal_signal : combined_score from internal activations in [0, 1]
                          0 = activations say grounded, 1 = activations say hallucinated
        internal_weight : how much to weight internal signals vs external evidence

        Returns continuous grounding score and verdict.
        """
        if not claim_evidences:
            external_groundedness = 0.5
            confidence = 0.0
        else:
            mean_evidence = float(np.mean(claim_evidences))
            external_groundedness = (mean_evidence + 1.0) / 2.0
            evidence_consistency = 1.0 - float(np.std(claim_evidences))
            confidence = float(np.tanh(2.0 * abs(mean_evidence))) * evidence_consistency

        internal_groundedness = 1.0 - internal_signal
        ext_weight = 1.0 - internal_weight
        final_groundedness = (
            ext_weight * external_groundedness +
            internal_weight * internal_groundedness
        )
        hallucination_score = 1.0 - final_groundedness
        agreement = 1.0 - abs(external_groundedness - internal_groundedness)

        if agreement > 0.7:
            if hallucination_score > 0.5:
                verdict = "HIGH_RISK"
            else:
                verdict = "LOW_RISK"
        else:
            if confidence > 0.5:
                if hallucination_score > 0.5:
                    verdict = "HIGH_RISK"
                else:
                    verdict = "LOW_RISK"
            else:
                verdict = "UNCERTAIN"

        return {
            "hallucination_score":   hallucination_score,
            "final_groundedness":    final_groundedness,
            "external_groundedness": external_groundedness,
            "internal_groundedness": internal_groundedness,
            "agreement":             agreement,
            "confidence":            confidence,
            "verdict":               verdict,
        }


# ════════════════════════════════════════════════════════════════════════════
# SUPPORTING CLASSES — kept from v1, used as internal signal stream
# ════════════════════════════════════════════════════════════════════════════

class ReadRecallScorer:
    def score(self, f: dict) -> float:
        return float(1.0 / (1.0 + np.exp(float(f.get("pks_late", 0.0))
                                         - float(f.get("ecs_mean", 0.0)))))


class PatternSeparator:
    def score(self, f: dict) -> tuple[float, float]:
        sep    = float(0.5 + 0.5 * np.tanh(float(f.get("cdr_q_delta", 0.0)) * 3.0))
        schema = float(0.5 * (1.0 + np.tanh(float(f.get("uptake", 0.0)) / 3.0 - 1.0)))
        return sep, schema


class ContextLengthBaseline:
    def __init__(self):
        kernel = (RBF(length_scale=100.0, length_scale_bounds=(10.0,1e4))
                  + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-3,1.0)))
        self.gpr    = GaussianProcessRegressor(
            kernel=kernel, alpha=1e-5,
            normalize_y=True, n_restarts_optimizer=3
        )
        self.fitted = False

    def fit(self, X, prompt_lengths, groups):
        target = X[:, :len(cfg.analysis_layers)].mean(axis=1)
        try:
            self.gpr.fit(prompt_lengths.reshape(-1,1).astype(float), target)
            self.fitted = True
            print("ContextLengthBaseline GPR fitted ✓")
        except Exception as e:
            print(f"ContextLengthBaseline fit failed: {e}")

    def compute_error(self, X, prompt_len):
        if not self.fitted: return 0.0
        observed = float(X[:, :len(cfg.analysis_layers)].mean())
        try:
            mu, sigma = self.gpr.predict(
                np.array([[float(prompt_len)]]), return_std=True
            )
            return float((observed - float(mu[0])) / max(float(sigma[0]), 1e-6))
        except:
            return 0.0


class SignalEnsemble:
    def __init__(self):
        self.specialists  = {}
        self.scaler       = StandardScaler()
        self.feature_groups = {}
        self.fitted       = False

    def fit(self, X, y, groups):
        self.feature_groups = groups
        X_sc = self.scaler.fit_transform(X)
        self.specialists.clear()
        for name, idxs in groups.items():
            clf = LogisticRegression(
                C=0.1, max_iter=1000,
                class_weight="balanced", random_state=42
            )
            try:
                clf.fit(X_sc[:, idxs], y)
                self.specialists[name] = clf
            except Exception as e:
                print(f"  Specialist {name} failed: {e}")
        self.fitted = True
        print(f"SignalEnsemble: {len(self.specialists)} specialists fitted ✓")

    def vote(self, X):
        if not self.fitted or not self.specialists: return 0.5, {}
        X_sc = self.scaler.transform(X); probs = {}
        for name, clf in self.specialists.items():
            idxs = self.feature_groups.get(name, list(range(X_sc.shape[1])))
            try:
                probs[name] = float(clf.predict_proba(X_sc[:, idxs])[0,1])
            except:
                probs[name] = 0.5
        return float(np.mean(list(probs.values()))), probs


class OnlineEnsembleUpdater:
    BUFFER_CAPACITY = 50

    def __init__(self, ensemble):
        self.ensemble = ensemble
        self.X_buffer = []
        self.y_buffer = []

    def add(self, X, score):
        if score > 0.8:
            self.X_buffer.append(X.flatten().copy())
            self.y_buffer.append(1)
        elif score < 0.2:
            self.X_buffer.append(X.flatten().copy())
            self.y_buffer.append(0)

    def consolidate(self):
        if len(self.X_buffer) < self.BUFFER_CAPACITY: return None
        X = np.array(self.X_buffer); y = np.array(self.y_buffer)
        self.X_buffer.clear(); self.y_buffer.clear()
        print(f"[SWR] Consolidation: {len(y)} samples "
              f"({y.sum()} hal, {(1-y).sum()} faithful)")
        return X, y


# ════════════════════════════════════════════════════════════════════════════
# RAG HALLUCINATION DETECTOR v2
# Orchestrates all four layers + internal activation signals.
# ════════════════════════════════════════════════════════════════════════════

class RAGHallucinationDetector:
    """
    Claim-level evidence integration detector.
    Combines external evidence verification (claims vs document spans)
    with internal activation signals (CDR delta, uptake, ECS, etc).
    """

    def __init__(self):
        self.decomposer  = ClaimDecomposer()
        self.retriever   = EvidenceRetriever(semantic_model)
        self.span_nli    = SpanLevelNLI(nli_model_hf, nli_tokenizer)
        self.integrator  = EvidenceIntegrator()
        self.rr          = ReadRecallScorer()
        self.dc          = PatternSeparator()
        self.baseline    = ContextLengthBaseline()
        self.ens         = SignalEnsemble()
        self.updater     = None

    def fit(self, X, y, groups, prompt_lengths):
        self.ens.fit(X, y, groups)
        self.baseline.fit(X, prompt_lengths, groups)
        self.updater = OnlineEnsembleUpdater(self.ens)

    def verify_claims(self, answer: str, document: str) -> dict:
        """
        Layer 1-3: decompose answer, retrieve evidence, score grounding.
        Returns aggregated claim verification result.
        """
        if self.decomposer.is_refusal(answer):
            return {
                "is_refusal":          True,
                "claims":              [],
                "evidence_strengths":  [],
                "external_groundedness": 1.0,
                "claim_details":       [],
            }

        claims = self.decomposer.decompose(answer)
        if not claims:
            return {
                "is_refusal":          False,
                "claims":              [],
                "evidence_strengths":  [],
                "external_groundedness": 0.5,
                "claim_details":       [],
            }

        evidence_strengths = []
        claim_details = []

        for claim in claims:
            evidence_span, retrieval_sim = self.retriever.retrieve(claim, document)
            nli_result = self.span_nli.score(claim, evidence_span)
            strength   = self.integrator.claim_evidence_strength(
                nli_result, retrieval_sim
            )
            evidence_strengths.append(strength)
            claim_details.append({
                "claim":         claim,
                "evidence":      evidence_span,
                "retrieval_sim": retrieval_sim,
                "entailment":    nli_result["entailment"],
                "contradiction": nli_result["contradiction"],
                "strength":      strength,
            })

        mean_strength       = float(np.mean(evidence_strengths))
        external_groundedness = (mean_strength + 1.0) / 2.0
        return {
            "is_refusal":            False,
            "claims":                claims,
            "evidence_strengths":    evidence_strengths,
            "external_groundedness": external_groundedness,
            "claim_details":         claim_details,
        }

    def predict(self, X: np.ndarray, f: dict, prompt_len: int,
                tok_re=None, doc: str = "", question: str = "",
                answer: str = "") -> dict:
        """
        Main entry point.
        Args:
            X           : feature vector (shape (1, feature_dim))
            f           : dict with 'ca1_match', 'ca1_mismatch', 'ecs_mean',
                          'pks_late', 'uptake', 'cdr_q_delta', and 'answer'
            prompt_len  : length of the prompt
            tok_re      : tokenised prompt (unused now, kept for compat)
            doc         : the document
            question    : the question
            answer      : the model's generated answer (REQUIRED for v2)
        """

        rr             = self.rr.score(f)
        sep, schema    = self.dc.score(f)
        baseline_err_z = self.baseline.compute_error(X, prompt_len)
        ens_score, votes = self.ens.vote(X)

        gain            = float(1.0 / (1.0 + np.exp(baseline_err_z)))
        nli_drive_old   = float(f.get("ca1_mismatch", 0.5)) - float(f.get("ca1_match", 0.5))
        modulated_drive = nli_drive_old * gain
        internal_term   = 0.5 + 0.5 * modulated_drive

        internal_signal = float(
            0.40 * ens_score +
            0.30 * internal_term +
            0.15 * (1.0 - rr) +
            0.15 * (1.0 - sep)
        )

        if answer and doc:
            verification = self.verify_claims(answer, doc)
        else:
            verification = {
                "is_refusal":            False,
                "claims":                [],
                "evidence_strengths":    [],
                "external_groundedness": 0.5,
                "claim_details":         [],
            }

        if verification["is_refusal"]:
            integration = {
                "hallucination_score":   0.0,
                "final_groundedness":    1.0,
                "external_groundedness": 1.0,
                "internal_groundedness": 1.0 - internal_signal,
                "agreement":             1.0,
                "confidence":            1.0,
                "verdict":               "LOW_RISK",
            }
        else:
            integration = self.integrator.integrate(
                verification["evidence_strengths"],
                internal_signal,
                internal_weight=0.30,
            )

        if self.updater is not None:
            self.updater.add(X, integration["hallucination_score"])
            replay = self.updater.consolidate()
            if replay is not None:
                Xr, yr = replay
                if len(np.unique(yr)) > 1:
                    self.ens.fit(Xr, yr, self.ens.feature_groups)

        return {
            "verdict":              integration["verdict"],
            "combined_score":       integration["hallucination_score"],
            "external_groundedness": integration["external_groundedness"],
            "internal_groundedness": integration["internal_groundedness"],
            "agreement":            integration["agreement"],
            "confidence":           integration["confidence"],
            "n_claims":             len(verification["claims"]),
            "n_grounded":           sum(1 for s in verification["evidence_strengths"] if s > 0.2),
            "n_contradicted":       sum(1 for s in verification["evidence_strengths"] if s < -0.2),
            "is_refusal":           verification["is_refusal"],
            "claim_details":        verification["claim_details"],
            "internal_signal":      internal_signal,
            "ens_score":            ens_score,
            "rr_score":             rr,
            "sep_score":            sep,
            "baseline_err_z":       baseline_err_z,
            "gain":                 gain,
            "ca1_match":            float(f.get("ca1_match", 0.5)),
            "ca1_mismatch":         float(f.get("ca1_mismatch", 0.5)),
            "specialist_votes":     votes,
            "nli_drive":            nli_drive_old,
            "hal_drive":            integration["external_groundedness"],
            "faith_drive":          1.0 - integration["external_groundedness"],
            "inhibition_margin":    1.0 - 2.0 * integration["final_groundedness"],
            "source_verdict":       "claim_verification",
            "retrieval_heads_fired": 0,
            "n_retrieval_heads":     0,
            "source_score":          integration["confidence"],
        }


# ── Backward-compat alias so old code referencing ca1_comparator still works
def answer_doc_comparator(answer: str, document: str) -> tuple[float, float]:
    """Compatibility shim. Returns (match, mismatch) from claim verification."""
    detector_temp = RAGHallucinationDetector()
    v = detector_temp.verify_claims(answer, document)
    g = v["external_groundedness"]
    return float(g), float(1.0 - g)


# ════════════════════════════════════════════════════════════════════════════
# INSTANTIATE
# ════════════════════════════════════════════════════════════════════════════

n_l = len(cfg.analysis_layers)
feature_groups = {
    "cdr_specialist":    list(range(0,       2 * n_l)),
    "ecs_specialist":    list(range(2 * n_l, 3 * n_l)),
    "pks_specialist":    list(range(3 * n_l, 4 * n_l)),
    "uptake_specialist": [4 * n_l],
    "mmd_specialist":    [4 * n_l + 1],
    "token_specialist":  [4 * n_l + 3],
    "cg_specialist":     [4 * n_l + 4],
}

print("Building Claim-Level Evidence Integration Detector v2...")
detector = RAGHallucinationDetector()
prompt_lengths = np.array([[len(s.prompt)] for s in dataset])
detector.fit(X_sota, y_sota, feature_groups, prompt_lengths)
print("Detector ready ✓")

In [ ]:
def make_feats_dict(X_row: np.ndarray, fg: dict,
                    answer: str, document: str) -> dict:
    ca1_m, ca1_mm   = answer_doc_comparator(answer, document)
    cdr_q_delta_raw = float(X_row[n_l : 2*n_l].mean())
    ecs_raw         = float(X_row[fg["ecs_specialist"]].mean())
    pks_late_raw    = float(X_row[fg["pks_specialist"]][-5:].mean())
    uptake_raw      = float(X_row[fg["uptake_specialist"][0]])
    mmd_raw         = float(X_row[fg["mmd_specialist"][0]])
    return {
        "ca1_match": ca1_m, "ca1_mismatch": ca1_mm,
        "ecs_mean": ecs_raw, "cdr_q_delta": cdr_q_delta_raw,
        "pks_late": pks_late_raw, "uptake": uptake_raw, "mmd": mmd_raw,
    }


def chunked_detector(detector: RAGHallucinationDetector,
                            document: str, question: str,
                            chunk_size: int = 256, overlap: int = 64) -> dict:
    tokens = document.split()

    def run_chunk(doc_chunk: str):
        s = ConflictSample(
            prompt=(f"Read the following document carefully:\n"
                    f"DOCUMENT: {doc_chunk}\n\n"
                    f"Based ONLY on the document, answer:\n{question}\nAnswer:"),
            context_answer="", parametric_answer="",
            question=question, document=doc_chunk,
            category="user", source="user", label=-1,
        )
        tok_re = model.to_tokens(s.prompt, prepend_bos=True).to("cuda:0")
        out    = model.generate(tok_re, max_new_tokens=40,
                                prepend_bos=False, verbose=False)
        ans    = (model.tokenizer.decode(out[0][tok_re.shape[1]:])
                  .strip().split("\n")[0].strip()
                  .replace("<end_of_turn>","").replace("<eos>","")
                  .replace("<bos>","").replace("**","").strip()) or "[no output]"
        fv = extract_sota_features(model, s)
        if fv is None: return None
        fv  = np.nan_to_num(fv, nan=0.0, posinf=10.0, neginf=-10.0)
        fd  = make_feats_dict(fv, feature_groups, ans, doc_chunk)
        res = detector.predict(fv.reshape(1,-1), fd, len(s.prompt),tok_re=tok_re, doc=doc_chunk, question=question,answer=ans)        
        res["chunk_answer"] = ans
        print(f"  [chunk] ca1_m={fd['ca1_match']:.3f} "
              f"ca1_mm={fd['ca1_mismatch']:.3f} "
              f"score={res['combined_score']:.4f} → {res['verdict']}")
        return res

    if len(tokens) <= chunk_size:
        print(f"\n[CHUNKER] {len(tokens)} tokens — single chunk")
        out = run_chunk(document)
        if out is None:
            return {"verdict":"ERROR","n_chunks":1,"mean_score":0.5,
                    "n_high":0,"n_low":0,"n_unc":0}
        return {"verdict":out["verdict"],"mean_score":out["combined_score"],
                "chunks":[out],"n_chunks":1,
                "n_high":1 if out["verdict"]=="HIGH_RISK" else 0,
                "n_low": 1 if out["verdict"]=="LOW_RISK"  else 0,
                "n_unc": 1 if out["verdict"]=="UNCERTAIN" else 0}

    chunks = [" ".join(tokens[i:i+chunk_size])
              for i in range(0, len(tokens), chunk_size-overlap)]
    print(f"\n[CHUNKER] {len(tokens)} tokens → {len(chunks)} chunks")
    results = []
    for ci, ch in enumerate(chunks):
        print(f"  chunk {ci+1}/{len(chunks)}")
        out = run_chunk(ch)
        if out:
            out["chunk_idx"] = ci; out["chunk_preview"] = ch[:80]+"..."
            results.append(out)
        torch.cuda.empty_cache()

    if not results:
        return {"verdict":"ERROR","n_chunks":len(chunks),"mean_score":0.5,
                "n_high":0,"n_low":0,"n_unc":0}

    mean_score = float(np.array([r["combined_score"] for r in results]).mean())
    is_unc     = detector.mm.is_uncertain(mean_score)
    verdict    = ("UNCERTAIN" if is_unc else
                  "HIGH_RISK" if mean_score >= 0.5 else "LOW_RISK")
    n_high = sum(1 for r in results if r["verdict"]=="HIGH_RISK")
    n_low  = sum(1 for r in results if r["verdict"]=="LOW_RISK")
    n_unc  = sum(1 for r in results if r["verdict"]=="UNCERTAIN")
    print(f"[CHUNKER] mean={mean_score:.4f} → {verdict} "
          f"(H={n_high} L={n_low} U={n_unc} — display only)")
    return {"verdict":verdict,"mean_score":mean_score,"chunks":results,
            "n_chunks":len(results),"n_high":n_high,"n_low":n_low,"n_unc":n_unc}

print("Chunked detector ready ✓")



### 6. Interactive Diagnostic Dashboard
Launches a Gradio interface allowing users to input custom documents and questions. The dashboard provides a live, 0-10 continuous risk score alongside a breakdown of the specific internal activation drifts driving the verdict.

In [ ]:
import gradio as gr

with open(f"{cfg.output_dir}/sota_calibration.pkl","rb") as f:
    sota_calib = pickle.load(f)
sota_scaler = sota_calib["scaler"]; sota_clf = sota_calib["clf"]
sota_thr    = sota_calib["threshold"]; sota_auc = sota_calib["cv_auc"]
signal_aucs = sota_calib["signal_aucs"]; analysis_lr = sota_calib["analysis_layers"]
print(f"Calibration loaded: AUC={sota_auc:.3f} thr={sota_thr:.4f} "
      f"n={sota_calib['n_samples']}")


@torch.no_grad()
def analyze_query(context: str, question: str, device: str = "cuda:0"):
    if not context.strip() or not question.strip():
        return ("Please provide both a document and a question.", "⚠️ NO INPUT", "", None)

    chunk_result  = chunked_detector(detector, context, question, 256, 64)
    brain_verdict = chunk_result["verdict"]
    mean_score    = float(chunk_result.get("mean_score", 0.5))

    if brain_verdict == "HIGH_RISK":
        detector_label = "🚨 HIGH HALLUCINATION RISK — model answering from memory"
        risk_color  = "#D85A30"
    elif brain_verdict == "UNCERTAIN":
        detector_label = "⚠️ UNCERTAIN — verify answer manually"
        risk_color  = "#BA7517"
    else:
        detector_label = "✅ LOW RISK — model reading your document"
        risk_color  = "#1D9E75"

    chunk_note = ""
    if chunk_result["n_chunks"] > 1:
        chunk_note = f"\nLong doc: {chunk_result['n_chunks']} chunks | mean={mean_score:.4f}"

    last_preds  = chunk_result.get("chunks", [])
    last_pred   = last_preds[-1] if last_preds else {}
    nli_drive_v = float(last_pred.get("nli_drive",    0.0))
    gain_v      = float(last_pred.get("context_gain",0.5))
    baseline_err_z_v= float(last_pred.get("baseline_error_z", 0.0))
    ens_v      = float(last_pred.get("ensemble_score",0.5))
    rr_v        = float(last_pred.get("read_ratio",  0.5))
    sep_v        = float(last_pred.get("separation_score",     0.5))

    p_nc = f"Answer the following question:\n{question}\nAnswer:"
    p_re = (f"Read the following document carefully:\nDOCUMENT: {context}\n\n"
            f"Based ONLY on the document, answer:\n{question}\nAnswer:")
    rd   = random.choice([d for d in RANDOM_DOCS if d != context] or RANDOM_DOCS)
    p_ra = (f"Read the following document carefully:\nDOCUMENT: {rd}\n\n"
            f"Based ONLY on the document, answer:\n{question}\nAnswer:")

    tok_nc = model.to_tokens(p_nc, prepend_bos=True).to(device)
    tok_re = model.to_tokens(p_re, prepend_bos=True).to(device)
    tok_ra = model.to_tokens(p_ra, prepend_bos=True).to(device)
    qt     = model.to_tokens(question, prepend_bos=False)[0]
    ql     = len(qt)
    qp     = max(2, min(tok_re.shape[1]-ql-3, tok_re.shape[1]-4))
    qp_nc  = max(2, tok_nc.shape[1]-ql-3)

    def run_hooks(tokens):
        ac, mc, rp, rm = {}, {}, {}, {}
        def mka(l):
            def fn(v, **kw): ac[l]=v[0].detach().cpu().float(); return v
            return fn
        def mkm(l):
            def fn(v, **kw): mc[l]=v[0].detach().cpu().float(); return v
            return fn
        def mkp(l):
            def fn(v, **kw): rp[l]=v[0,-1,:].detach().cpu().float(); return v
            return fn
        def mkd(l):
            def fn(v, **kw): rm[l]=v[0,-1,:].detach().cpu().float(); return v
            return fn
        hooks = []
        for l in analysis_lr:
            hooks += [(f"blocks.{l}.hook_attn_out",  mka(l)),
                      (f"blocks.{l}.hook_mlp_out",   mkm(l)),
                      (f"blocks.{l}.hook_resid_pre", mkp(l)),
                      (f"blocks.{l}.hook_resid_mid", mkd(l))]
        with model.hooks(fwd_hooks=hooks): lg = model(tokens)
        return lg, ac, mc, rp, rm

    lg_nc, ac_nc, mc_nc, _,     _     = run_hooks(tok_nc)
    lg_re, ac_re, mc_re, rp_re, rm_re = run_hooks(tok_re)
    lg_ra, _,     _,     _,     _     = run_hooks(tok_ra)

    out = model.generate(tok_re, max_new_tokens=40, prepend_bos=False, verbose=False)
    ans = (model.tokenizer.decode(out[0][tok_re.shape[1]:])
           .strip().split("\n")[0].strip()
           .replace("<end_of_turn>","").replace("<eos>","")
           .replace("<bos>","").strip() or "[no output]")

    nli_contradiction_prob = 0.5
    if nli_model_hf is not None and nli_tokenizer is not None:
        try:
            doc_trunc_ui = " ".join(context.split()[:400])
            inputs_ui = nli_tokenizer(doc_trunc_ui, ans, return_tensors="pt",
                                      truncation=True, max_length=512).to(device)
            with torch.no_grad():
                probs_ui = nli_model_hf(**inputs_ui).logits.softmax(-1)[0]
            nli_contradiction_prob = (float(probs_ui[0].item())
                                      if probs_ui.shape[0]==2
                                      else float(probs_ui[2].item()))
        except Exception as e:
            print(f"  [UI NLI] failed: {e}")

    pn = lg_nc[0,-1].float().softmax(-1)
    pr = lg_re[0,-1].float().softmax(-1)
    pa = lg_ra[0,-1].float().softmax(-1)

    uptake = float(F.kl_div(pn.clamp(1e-10).log(), pr.clamp(1e-10), reduction="sum").item())
    mmd    = float(0.5*(
        F.kl_div(pa.clamp(1e-10).log(), pr.clamp(1e-10), reduction="sum").item() +
        F.kl_div(pr.clamp(1e-10).log(), pa.clamp(1e-10), reduction="sum").item()
    ))
    top_ov = len(set(pr.topk(50).indices.tolist()) & set(pa.topk(50).indices.tolist())) / 50

    ecs = []
    for l in analysis_lr:
        if l not in ac_re: ecs.append(0.0); continue
        av = ac_re[l][-1]; cv = ac_re[l][:qp]
        ecs.append(float(F.cosine_similarity(av.unsqueeze(0), cv, dim=-1).mean().item())
                   if cv.shape[0]>0 else 0.0)
    ecs = np.array(ecs)

    pks = []
    for l in analysis_lr:
        if l not in rp_re or l not in rm_re: pks.append(0.0); continue
        pl = (rp_re[l].to(device) @ model.W_U.float()).softmax(-1).cpu()
        ml = (rm_re[l].to(device) @ model.W_U.float()).softmax(-1).cpu()
        mv = 0.5*(pl+ml)
        pks.append(float(0.5*(
            F.kl_div(mv.clamp(1e-10).log(), pl.clamp(1e-10), reduction="sum") +
            F.kl_div(mv.clamp(1e-10).log(), ml.clamp(1e-10), reduction="sum")
        ).item()))
    pks = np.array(pks)

    def cdr_at(ac, mc, pos):
        v = np.zeros(len(analysis_lr))
        for i, l in enumerate(analysis_lr):
            if l in ac and l in mc and ac[l].shape[0]>abs(pos):
                a = ac[l][pos].norm().item(); m = mc[l][pos].norm().item()
                v[i] = a/(a+m+1e-8)
        return v
    da = cdr_at(ac_re, mc_re, -1)  - cdr_at(ac_nc, mc_nc, -1)
    dq = cdr_at(ac_re, mc_re,  qp) - cdr_at(ac_nc, mc_nc, qp_nc)

    def top_tid(lg):
        for t in lg[0,-1].topk(20).indices.tolist():
            d = model.tokenizer.decode([t]).strip().replace("▁","").replace("*","").strip()
            if d and d.lower() not in ["","<eos>","<bos>","<end_of_turn>"]: return t, d
        return -1, "[empty]"
    tnc, snc = top_tid(lg_nc); tre, sre = top_tid(lg_re)
    tc = 1.0 if tnc != tre else 0.0
    ctx_t = model.to_tokens(context.split(".")[0], prepend_bos=False)[0]
    cg    = max((float((pr[t.item()]-pn[t.item()]).item())
                 for t in ctx_t[:5] if 0<=t.item()<pr.shape[0]), default=0.0)

    feats = np.concatenate([da, dq, ecs, pks,
                             [uptake],[mmd],[top_ov],[tc],[cg],
                             [da.mean()],[np.abs(da).max()],[dq.mean()],
                             [ecs.mean()],[pks.mean()],[pks[-5:].mean()]])
    td    = sota_calib["feature_dim"]
    feats = np.nan_to_num(np.pad(feats,(0,max(0,td-len(feats))))[:td],
                          nan=0.0, posinf=10.0, neginf=-10.0)
    sota_score = float(sota_clf.predict_proba(
        sota_scaler.transform(feats.reshape(1,-1)))[0,1])

    ecs_m  = float(ecs.mean()); pks_lm = float(pks[-5:].mean())
    el = [i for i,l in enumerate(analysis_lr) if 4 <=l<=7 ]
    kl = [i for i,l in enumerate(analysis_lr) if 8 <=l<=9 ]
    ll = [i for i,l in enumerate(analysis_lr) if 19<=l<=25]
    em = float(da[el].mean()) if el else 0.0
    km = float(da[kl].mean()) if kl else 0.0
    lm = float(da[ll].mean()) if ll else 0.0

    explanation = (
        f"RAG HALLUCINATION DETECTOR — SIGNAL BREAKDOWN\n{'='*60}\n"
        f"  Detector verdict      : {brain_verdict}\n"
        f"  Mean chunk score   : {mean_score:.6f}  (→ 1 = hallucination)\n"
        f"  SOTA score (ref)   : {sota_score:.6f}\n"
        f"  CV AUC             : {sota_auc:.3f} (5-fold, n={sota_calib['n_samples']})"
        f"{chunk_note}\n\n"
        f"CONTINUOUS SIGNAL STATE\n{'-'*60}\n"
        f"  NLI drive          : {nli_drive_v:+.6f}\n"
        f"  Context gain       : {gain_v:.6f}  (σ(-Z))\n"
        f"  Baseline error Z   : {baseline_err_z_v:+.6f}  (GPR Z-score)\n"
        f"  Ensemble score     : {ens_v:.6f}  (mean P(hallucination))\n"
        f"  Read ratio (RR)    : {rr_v:.6f}  (σ(ECS-PKS))\n"
        f"  Pattern sep        : {sep_v:.6f}  (tanh(CDR_q×3))\n\n"
        f"  Hal drive          : {last_pred.get('hal_drive', 0.0):.6f}\n"
        f"  Faith drive        : {last_pred.get('faith_drive', 0.0):.6f}\n"
        f"  Inhibition margin  : {last_pred.get('inhibition_margin', 0.0):+.6f}  (+ = hal wins)\n"
        f"  Induction heads    : {last_pred.get('retrieval_heads_fired', 0)}"
        f"/{last_pred.get('n_retrieval_heads', 0)} fired on document\n"
        f"  Source verdict     : {last_pred.get('source_verdict', 'unknown')}\n"
        f"RAW MECHANISTIC SIGNALS\n{'-'*60}\n"
        f"  [PCIB]    Uptake (KL nats)   : {uptake:.6f}\n"
        f"  [LUMINA]  MMD (sym-KL nats)  : {mmd:.6f}\n"
        f"  [LUMINA]  Top-50 overlap     : {top_ov:.6f}\n"
        f"  [ReDeEP]  ECS mean           : {ecs_m:+.6f}\n"
        f"  [ReDeEP]  PKS late           : {pks_lm:.6f}\n"
        f"  [TwoPath] Q-anchor CDR Δ     : {dq.mean():+.6f}\n"
        f"  [TwoPath] A-anchor CDR Δ     : {da.mean():+.6f}\n"
        f"  [NLI]     Contradiction P    : {nli_contradiction_prob:.6f}\n"
        f"  Token shift                  : '{snc}'→'{sre}' (tc={tc:.1f})\n\n"
        f"THREE-PHASE CDR BREAKDOWN (L4-7 / L8-9 / L19+)\n{'-'*60}\n"
        f"  Phase 1 — Separation  L4-7    : {em:+.6f}\n"
        f"  Phase 2 — Memory check L8-9   : {km:+.6f}\n"
        f"  Phase 3 — Commitment  L19+    : {lm:+.6f}\n\n"
        f"MODEL ANSWER: {ans}"
    )

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=["Signals (tanh-normalized for display)",
                        "ReDeEP ECS (CKA) + PKS (JS-div) per layer",
                        "Three-Phase CDR Δ",
                        f"Combined Score = {mean_score:.4f}"],
        specs=[[{"type":"xy"},{"type":"xy"}],[{"type":"xy"},{"type":"indicator"}]],
        vertical_spacing=0.20, horizontal_spacing=0.12,
    )

    sn = ["PCIB\nUptake","LUMINA\nMMD","ReDeEP\nECS","ReDeEP\nPKS",
          "Q-CDR","A-CDR","Token\nChange","NLI\nContra"]
    sv = [float(np.tanh(uptake/3.0)),
          float(np.tanh(mmd/3.0)),
          float(0.5+0.5*np.tanh(ecs_m/0.2)),
          float(np.tanh(pks_lm/1.0)),
          float(0.5+0.5*np.tanh(dq.mean()/0.2)),
          float(0.5+0.5*np.tanh(da.mean()/0.2)),
          float(tc), float(nli_contradiction_prob)]
    av = [signal_aucs.get(k,0.5) for k in [
        "uptake","mmd","ecs","pks",
        "cdr_delta_question","cdr_delta_answer","token_changed","confidence_gain"]]

    fig.add_trace(go.Bar(x=sn, y=sv, name="Signal (tanh)",
        marker_color=["#1D9E75" if v>0.55 else "#BA7517" if v>0.45 else "#D85A30" for v in sv],
        text=[f"{v:.3f}" for v in sv], textposition="outside", textfont=dict(size=8),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(x=sn, y=av, mode="lines+markers+text", name="Per-signal AUC",
        line=dict(color="#7F77DD", width=2, dash="dot"), marker=dict(size=8,symbol="diamond"),
        text=[f"{a:.2f}" for a in av], textposition="top center",
        textfont=dict(size=8,color="#7F77DD"),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(x=analysis_lr, y=ecs.tolist(), name="ECS (CKA-aligned)",
        line=dict(color="#1D9E75",width=2), mode="lines+markers", marker=dict(size=4),
    ), row=1, col=2)
    fig.add_trace(go.Scatter(x=analysis_lr, y=pks.tolist(), name="PKS (JS-div)",
        line=dict(color="#D85A30",width=2), mode="lines+markers", marker=dict(size=4),
    ), row=1, col=2)
    fig.add_vrect(x0=20,x1=26,fillcolor="#D85A30",opacity=0.08,
        annotation_text="Knowledge FFNs",annotation_font_size=9,row=1,col=2)

    bc = []
    for i, l in enumerate(analysis_lr):
        d = float(da[i])
        bc.append("#1D9E75" if ((4<=l<=7 and d>0) or (8<=l<=9 and d<0) or (19<=l<=25 and d>0))
                  else "#D85A30" if ((4<=l<=7 and d<=0) or (8<=l<=9 and d>=0) or (19<=l<=25 and d<=0))
                  else "#9E9E9E")
    fig.add_trace(go.Bar(x=[f"L{l}" for l in analysis_lr], y=da.tolist(), marker_color=bc,
        name="CDR Δ (ctx-par)", text=[f"{d:+.3f}" for d in da],
        textposition="outside", textfont=dict(size=7),
    ), row=2, col=1)
    fig.add_hline(y=0, line_color="black", line_width=1, row=2, col=1)
    for x0,x1,col,lbl in [(2,5,"#1D9E75","Ph1"),(6,8,"#D85A30","Ph2"),(17,23,"#1D9E75","Ph3")]:
        fig.add_vrect(x0=x0,x1=x1,fillcolor=col,opacity=0.08,
            annotation_text=lbl,annotation_font_size=8,row=2,col=1)

    fig.add_trace(go.Indicator(
        mode="gauge+number", value=mean_score,
        title={"text":f"Mean chunk score<br><sub>{brain_verdict}</sub>","font":{"size":11}},
        gauge={"axis":{"range":[0,1]},"bar":{"color":risk_color,"thickness":0.3},
               "steps":[{"range":[0.00,0.35],"color":"#e8f5f0"},
                        {"range":[0.35,0.65],"color":"#fff8e1"},
                        {"range":[0.65,1.00],"color":"#fde8e0"}],
               "threshold":{"line":{"color":"#333","width":2},"thickness":0.75,"value":sota_score}},
        number={"valueformat":".4f","font":{"size":34}},
    ), row=2, col=2)

    fig.update_layout(
        title={"text":(f"RAG Monitor — {detector_label}<br>"
                       f"<sub>score={mean_score:.4f} | NLI={nli_contradiction_prob:.3f} | "
                       f"ECS={ecs_m:+.3f} | uptake={uptake:.2f}nat | mmd={mmd:.2f}nat | "
                       f"gain={gain_v:.3f} | Z={baseline_err_z_v:+.2f} | SOTA={sota_score:.3f}</sub>"),
               "font":{"size":11}},
        template="plotly_white", height=740, showlegend=True,
        legend=dict(x=0,y=-0.05,orientation="h",font=dict(size=9)),
    )
    fig.update_xaxes(tickangle=0,  tickfont=dict(size=9), row=1, col=1)
    fig.update_xaxes(title_text="Layer",                  row=1, col=2)
    fig.update_xaxes(tickangle=45, tickfont=dict(size=8), row=2, col=1)
    fig.update_yaxes(title_text="Signal (tanh)",          row=1, col=1)
    fig.update_yaxes(title_text="Raw score",              row=1, col=2)
    fig.update_yaxes(title_text="CDR Δ (raw)",            row=2, col=1)

    return ans, detector_label, explanation, fig


EXAMPLES = [
    ["All sales final. No returns or refunds accepted.",              "Can I return this?"],
    ["No discounts. No bereavement fares. Full price only.",         "Bereavement discount?"],
    ["Capital of Australia relocated to Sydney in 2024.",            "What is Australia's capital?"],
    ["Patient: 250mg once daily. Standard dose does NOT apply.",     "Correct dosage?"],
    ["Jones v. Smith (2019) only. No other precedents.",             "Relevant case law?"],
    ["Eiffel Tower in Paris, France. Built 1889.",                   "Where is the Eiffel Tower?"],
    ["Return policy: 30 days, unused items, full refund.",           "Return window?"],
    ["Cancellation: 90 days notice + $500 fee.",                     "How to cancel?"],
    ["Office hours: Mon-Fri 9am-6pm. Closed weekends.",              "Open Saturday?"],
    ["Conference: keynote 9am, panels 10am-12pm, workshops 1-4pm.", "When is keynote?"],
]

with gr.Blocks(theme=gr.themes.Soft(), title="RAG Hallucination Monitor") as demo:
    gr.Markdown(f"""
# 🔬 RAG Hallucination Monitor
Detects when a model ignores your document and answers from memory.
""")
    with gr.Row():
        with gr.Column(scale=1):
            ci  = gr.Textbox(label="📄 RAG Document", lines=8,
                             placeholder="Paste document — short or long...")
            qi  = gr.Textbox(label="❓ Question", lines=2)
            btn = gr.Button("🔬 Analyze",
                            variant="primary", size="lg")
            oa  = gr.Textbox(label="Model Answer", lines=3)
            or_ = gr.Textbox(label="Detector Verdict", lines=1)
            oe  = gr.Textbox(label="Signal Breakdown", lines=32)
        with gr.Column(scale=2):
            of = gr.Plot(label="Signal Dashboard — raw & tanh-normalized")
    gr.Examples(examples=EXAMPLES, inputs=[ci, qi],
                label="Top 5 = HIGH RISK | Bottom 5 = LOW RISK")
    btn.click(fn=analyze_query, inputs=[ci, qi], outputs=[oa, or_, oe, of])

demo.launch(share=True, debug=True)